In [1]:
import numpy as np

p = 80.0
k = 5

# 1. Prepare Probabilities
logits = np.array([5, 10, 6, 20, 7, 14, 3, 2, 13], dtype=float)
# Note: Real softmax uses exponents, but we'll stick to your linear sum method for this exercise
sum_logits = np.sum(logits)
softmax = logits / sum_logits
prob_dist = softmax * 100

# 2. Sorting (The crucial step for both K and P)
ranked_prob_dist = np.sort(prob_dist)[::-1]
print(f"Ranked Prob Dist: {ranked_prob_dist}")

# --- TOP-K Implementation ---
# Simply take the first k elements
top_k_probs = ranked_prob_dist[:k]
print(f"Top-K ({k}): {top_k_probs}")

# --- TOP-P Implementation ---
top_p_list = []
cumulative_sum = 0.0

for prob in ranked_prob_dist:
    top_p_list.append(prob)
    cumulative_sum += prob
    # Once the sum hits or exceeds p, we stop adding new tokens
    if cumulative_sum >= p:
        break

top_p = np.array(top_p_list)
print(f"Top-P ({p}%): {top_p}")
print(f"Final Cumulative Sum: {np.sum(top_p):.2f}%")


Ranked Prob Dist: [25.   17.5  16.25 12.5   8.75  7.5   6.25  3.75  2.5 ]
Top-K (5): [25.   17.5  16.25 12.5   8.75]
Top-P (80.0%): [25.   17.5  16.25 12.5   8.75]
Final Cumulative Sum: 80.00%


In [3]:
import numpy as np

p = 80.0  # Threshold percentage
k = 5     # Top-K count

logits = np.array([5, 10, 6, 20, 7, 14, 3, 2, 13], dtype=float)

# 1. Correct Softmax (with Numerical Stability)
# Subtracting max(logits) prevents overflow errors with np.exp
exp_logits = np.exp(logits - np.max(logits)) 
softmax = exp_logits / np.sum(exp_logits)
prob_dist = softmax * 100

# 2. Sorting (Essential for both K and P)
# We sort descending so the most likely tokens are first
sorted_indices = np.argsort(prob_dist)[::-1]
ranked_prob_dist = prob_dist[sorted_indices]

print(f"Full Ranked Probabilities (%):\n{np.round(ranked_prob_dist, 2)}\n")

# --- Correct TOP-K ---
top_k_probs = ranked_prob_dist[:k]
print(f"Top-K (k={k}): {np.round(top_k_probs, 2)}")

# --- Correct TOP-P (Nucleus) ---
# We use a loop that includes the token that pushes us over the limit
top_p_list = []
cumulative_sum = 0.0

for prob in ranked_prob_dist:
    top_p_list.append(prob)
    cumulative_sum += prob
    # The moment we hit or exceed 'p', we stop, but keep the current 'prob'
    if cumulative_sum >= p:
        break

top_p = np.array(top_p_list)
print(f"Top-P (p={p}%): {np.round(top_p, 2)}")
print(f"Final Cumulative Sum: {cumulative_sum:.2f}%")


Full Ranked Probabilities (%):
[9.966e+01 2.500e-01 9.000e-02 0.000e+00 0.000e+00 0.000e+00 0.000e+00
 0.000e+00 0.000e+00]

Top-K (k=5): [9.966e+01 2.500e-01 9.000e-02 0.000e+00 0.000e+00]
Top-P (p=80.0%): [99.66]
Final Cumulative Sum: 99.66%


The reason you see such a massive difference is that the **Exponential Softmax** (the standard way) and your **Linear Sum** (the simple way) treat the "distance" between numbers very differently.

### **The Comparison**


| Method | Logit 20 vs Logit 14 | Result |
| :--- | :--- | :--- |
| **Linear (Your initial code)** | $20$ is only $\approx 1.4\times$ larger than $14$. | The probabilities stay close together, so Top-P needs **5 tokens** to hit 80%. |
| **Exponential (Standard)** | $e^{20}$ is $\approx 403\times$ larger than $e^{14}$. | The largest number "eats" the whole distribution, so Top-P hits 80% with **1 token**. |

---

### **What you were missing: "Probability Mass"**

In your linear calculation:
1. Sum of logits: $5+10+6+20+7+14+3+2+13 = \mathbf{80}$.
2. The highest probability (for 20) is $20 / 80 = \mathbf{25\%}$.
3. Because the highest value is only 25%, the Top-P loop **must** keep grabbing the next tokens ($17.5\%, 16.25\%$, etc.) until it crawls its way up to the 80% goal.

In the exponential calculation:
1. $e^{20}$ is **485,165,195**.
2. $e^{14}$ is **1,202,604**.
3. The sum of all $e^{logits}$ is dominated by the 485 million. Thus, the 20 logit becomes **99.6%** of the total.
4. Since $99.6\% > 80.0\%$, the Top-P loop stops immediately.

### **Which one is "Correct"?**
In the context of Large Language Models (LLMs), the **Exponential version** is the only "correct" one. This is because the model's last layer produces **Logits** (log-odds), which are mathematically designed to be exponentiated according to the [Softmax function](https://en.wikipedia.org).

If the model is very confident (like with your logit of 20), the Exponential Softmax ensures the model doesn't "hallucinate" by picking a much weaker word.

### **The "Fix" for Testing**
If you want the **Exponential** version to behave more like your **Linear** version (where multiple tokens get picked), you use **Temperature**:
* If you set `temp = 10.0` in the exponential code, the logit 20 becomes $2.0$ and 14 becomes $1.4$. 
* Now $e^{2.0}$ is only $\approx 1.8\times$ larger than $e^{1.4}$.
* **Result:** The probabilities will spread out, and Top-P will return multiple tokens again.


# Updated Code with Temperature

What will happen now?
With temp = 5.0, the gap between the logit 20 and 14 is reduced. You should see the first probability drop from 99% down to roughly 50-60%, which will force the Top-P loop to collect multiple tokens to reach the 80% goal.

Try running this with temp = 5.0 to see how it forces the probabilities to spread out:


In [1]:
import numpy as np

p = 80.0
k = 5
temp = 5.0 # Increased temperature to flatten the distribution
# temp = 0.5 # Decreased temperature to sharpen the distribution

logits = np.array([5, 10, 6, 20, 7, 14, 3, 2, 13], dtype=float)

# 1. Apply Temperature then Softmax
# Dividing by temp makes the differences between 20 and 14 smaller
scaled_logits = logits / temp
exp_logits = np.exp(scaled_logits - np.max(scaled_logits)) 
softmax = exp_logits / np.sum(exp_logits)
prob_dist = softmax * 100

# 2. Sorting
sorted_indices = np.argsort(prob_dist)[::-1]
ranked_prob_dist = prob_dist[sorted_indices]

print(f"Temperature: {temp}")
print(f"Ranked Probabilities (%):\n{np.round(ranked_prob_dist, 2)}\n")

# 3. Top-K
top_k = ranked_prob_dist[:k]
print(f"Top-K (k={k}): {np.round(top_k, 2)}")

# 4. Top-P
top_p_list = []
cumulative_sum = 0.0
for prob in ranked_prob_dist:
    top_p_list.append(prob)
    cumulative_sum += prob
    if cumulative_sum >= p:
        break

print(f"Top-P (p={p}%): {np.round(top_p_list, 2)}")
print(f"Final Cumulative Sum: {cumulative_sum:.2f}%")


Temperature: 5.0
Ranked Probabilities (%):
[51.85 15.62 12.79  7.02  3.85  3.15  2.58  1.73  1.42]

Top-K (k=5): [51.85 15.62 12.79  7.02  3.85]
Top-P (p=80.0%): [51.85 15.62 12.79]
Final Cumulative Sum: 80.25%
